# 03 — Vector Database: Chroma, Qdrant, dan pgvector

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/07-embeddings-vectordb/03_vector_db.ipynb)

FAISS (notebook 02) hanyalah **index** di memori. Aplikasi nyata butuh lebih dari itu, dan di situlah **vector database** berperan:

| Kebutuhan | FAISS | Vector DB |
|---|---|---|
| Pencarian ANN | ✅ | ✅ (umumnya HNSW) |
| Menyimpan teks asli & **metadata** | ❌ | ✅ |
| **Filter** metadata (misalnya hanya ulasan negatif) | ❌ | ✅ |
| Tambah/ubah/hapus data (CRUD), persistensi ke disk | terbatas | ✅ |
| Server, API, multi-user, replikasi | ❌ | ✅ |

Kita mencoba tiga pilihan populer (semuanya open-source):
- **Chroma** (Apache-2.0): ringan, embedded di Python, cocok untuk prototipe dan RAG sederhana.
- **Qdrant** (Apache-2.0): fokus performa dan filter yang kaya; bisa berjalan lokal (di Python) atau sebagai server.
- **pgvector** (PostgreSQL License): ekstensi PostgreSQL, sehingga vektor tersimpan di database SQL yang sudah kamu kenal.

**Data:** NusaX-Senti ([IndoNLP/nusax](https://github.com/IndoNLP/nusax), lisensi **CC-BY-SA**), 1.000 kalimat ulasan/komentar. NusaX adalah dataset **paralel**: kalimat dengan `id` yang sama tersedia dalam bahasa Indonesia **dan** terjemahan bahasa Inggrisnya. Sifat ini kita manfaatkan untuk mengukur kualitas pencarian secara objektif.

**Model:** [`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) (Apache-2.0): model embedding kalimat **multibahasa** (50+ bahasa, termasuk Indonesia), 118 juta parameter, vektor 384 dimensi.

## 0. Cek GPU

Menghitung embedding jauh lebih cepat di GPU (**Runtime → Change runtime type → T4 GPU**). Untuk 1.000 kalimat di notebook ini, CPU pun masih cukup.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, berjalan di CPU (lebih lambat, tetapi cukup untuk notebook ini).")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Instalasi, Data, dan Embedding

Kita hitung embedding **sendiri** dengan sentence-transformers, lalu memasukkannya ke database. Dengan cara ini model yang dipakai jelas dan sama untuk kedua database.

In [ ]:
%pip install -q sentence-transformers chromadb qdrant-client pandas

Muat data NusaX (pasangan Indonesia–Inggris seperti notebook 01).

In [ ]:
import numpy as np
import pandas as pd

NUSAX_URL = "https://raw.githubusercontent.com/IndoNLP/nusax/main/datasets/sentiment"

def muat(bahasa):
    return pd.concat([pd.read_csv(f"{NUSAX_URL}/{bahasa}/{s}.csv") for s in ["train", "valid", "test"]])

df = muat("indonesian").merge(muat("english")[["id", "text"]], on="id", suffixes=("", "_en")).reset_index(drop=True)
print(f"{len(df)} pasangan kalimat | label: {df['label'].value_counts().to_dict()}")
df[["text", "text_en", "label"]].head(3)

Tambahkan metadata yang berguna untuk filter (jumlah kata), lalu encode semua kalimat bahasa Indonesia.

In [ ]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)
df["n_kata"] = df["text"].str.split().str.len()
emb = model.encode(df["text"].tolist(), normalize_embeddings=True, batch_size=64, show_progress_bar=False)
DIM = emb.shape[1]
print("embedding:", emb.shape)

## 2. Chroma

Konsep utama Chroma:
- **Client**: `PersistentClient(path)` menyimpan data ke folder, sehingga tetap ada setelah program ditutup.
- **Collection**: seperti tabel; berisi `ids`, `embeddings`, `documents` (teks asli), dan `metadatas`.
- Ruang jarak diatur di konfigurasi `hnsw` (`cosine`, `l2`, atau `ip`).

`embedding_function=None` berarti kita selalu menyerahkan embedding sendiri (Chroma tidak mengunduh model bawaannya).

In [ ]:
import shutil
import chromadb

CHROMA_DIR = "chroma_nusax"
shutil.rmtree(CHROMA_DIR, ignore_errors=True)      # mulai bersih setiap kali notebook dijalankan
client = chromadb.PersistentClient(path=CHROMA_DIR)
koleksi = client.create_collection("nusax", embedding_function=None, configuration={"hnsw": {"space": "cosine"}})

koleksi.add(
    ids=df["id"].astype(str).tolist(),
    embeddings=emb.tolist(),
    documents=df["text"].tolist(),
    metadatas=[{"label": r.label, "n_kata": int(r.n_kata)} for r in df.itertuples()],
)
print("jumlah dokumen:", koleksi.count())

### 2.1 Query dengan dan tanpa filter metadata

Filter `where` memakai operator seperti `$eq`, `$gte`, `$in`, `$and`. Chroma mengembalikan **jarak** cosine (`1 − similarity`), jadi semakin kecil semakin mirip.

In [ ]:
def tampilkan_chroma(hasil):
    for doc, jarak, meta in zip(hasil["documents"][0], hasil["distances"][0], hasil["metadatas"][0]):
        print(f"   [sim {1 - jarak:.2f} | {meta['label']:<8} | {meta['n_kata']:>2} kata] {doc[:100]}")


query = "pelayanan restoran yang mengecewakan"
q_emb = model.encode([query], normalize_embeddings=True).tolist()

print(f"🔎 {query} (tanpa filter)")
tampilkan_chroma(koleksi.query(query_embeddings=q_emb, n_results=3))

print(f"\n🔎 {query} (hanya label positive, minimal 15 kata)")
tampilkan_chroma(koleksi.query(query_embeddings=q_emb, n_results=3,
                               where={"$and": [{"label": "positive"}, {"n_kata": {"$gte": 15}}]}))

### 2.2 CRUD dan persistensi

Data bisa diubah (`update`/`upsert`) dan dihapus (`delete`). Karena memakai `PersistentClient`, data tetap ada saat client dibuka ulang.

In [ ]:
baru = "Pelayan restorannya ramah sekali dan makanannya cepat datang."
koleksi.upsert(ids=["baru-1"], documents=[baru], embeddings=model.encode([baru], normalize_embeddings=True).tolist(),
               metadatas=[{"label": "positive", "n_kata": len(baru.split())}])
print("setelah upsert :", koleksi.count())
koleksi.delete(ids=["baru-1"])
print("setelah delete :", koleksi.count())

client_baru = chromadb.PersistentClient(path=CHROMA_DIR)       # simulasi program dibuka ulang
print("dibuka ulang   :", client_baru.get_collection("nusax").count(), "dokumen masih ada")

## 3. Qdrant

Qdrant memakai istilah **point** (id + vektor + **payload**/metadata). `QdrantClient(":memory:")` menjalankan Qdrant di dalam proses Python tanpa server, cocok untuk belajar. Untuk produksi, kode yang sama tinggal diarahkan ke server: `QdrantClient(url=..., api_key=os.getenv("QDRANT_API_KEY"))`, dengan API key dari `.env`/Secrets, bukan ditulis di kode.

Qdrant mengembalikan **skor similarity** (semakin besar semakin mirip).

In [ ]:
from qdrant_client import QdrantClient, models

qd = QdrantClient(":memory:")
qd.create_collection("nusax", vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE))
qd.upsert("nusax", points=[
    models.PointStruct(id=int(r.id), vector=emb[i].tolist(), payload={"text": r.text, "label": r.label, "n_kata": int(r.n_kata)})
    for i, r in enumerate(df.itertuples())
])
print("jumlah point:", qd.count("nusax").count)


def cari_qdrant(query, k=3, label=None, min_kata=None):
    syarat = []
    if label:
        syarat.append(models.FieldCondition(key="label", match=models.MatchValue(value=label)))
    if min_kata:
        syarat.append(models.FieldCondition(key="n_kata", range=models.Range(gte=min_kata)))
    hasil = qd.query_points("nusax", query=model.encode([query], normalize_embeddings=True)[0].tolist(), limit=k,
                            query_filter=models.Filter(must=syarat) if syarat else None, with_payload=True)
    return hasil.points


for p in cari_qdrant(query):
    print(f"   [sim {p.score:.2f} | {p.payload['label']:<8}] {p.payload['text'][:100]}")

### 3.1 Apakah Chroma dan Qdrant memberi hasil yang sama?

Keduanya memakai HNSW (pendekatan) dengan embedding yang sama, sehingga top-k seharusnya hampir identik. Kita bandingkan untuk beberapa query dengan exact search NumPy sebagai acuan.

In [ ]:
queries = ["kurir lambat mengantar paket", "harga murah kualitas bagus", "sinyal internet sering hilang",
           "pemandangan pantai yang indah", "saya kecewa dengan layanan bank"]
for q in queries:
    qv = model.encode([q], normalize_embeddings=True)
    exact = set(df["id"].to_numpy()[np.argsort(-(emb @ qv[0]))[:5]].astype(str))
    chroma_ids = set(koleksi.query(query_embeddings=qv.tolist(), n_results=5)["ids"][0])
    qdrant_ids = {str(p.id) for p in cari_qdrant(q, k=5)}
    print(f"{q:<35} Chroma vs exact {len(chroma_ids & exact)}/5 | Qdrant vs exact {len(qdrant_ids & exact)}/5")

## 4. pgvector: Vektor di PostgreSQL

Jika aplikasi sudah memakai PostgreSQL, **pgvector** sering menjadi pilihan paling sederhana: vektor disimpan di kolom biasa, dan pencarian digabung dengan SQL (JOIN, WHERE, transaksi) yang sudah kamu kuasai. Menjalankan PostgreSQL di Colab cukup merepotkan, jadi di sini hanya contoh SQL-nya (bisa dicoba di PostgreSQL lokal atau layanan seperti Supabase/Neon):

```sql
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE ulasan (
    id        BIGINT PRIMARY KEY,
    teks      TEXT NOT NULL,
    label     TEXT,
    n_kata    INT,
    embedding vector(384)            -- dimensi harus sama dengan model
);

-- Index HNSW untuk cosine distance (operator <=>)
CREATE INDEX ON ulasan USING hnsw (embedding vector_cosine_ops) WITH (m = 16, ef_construction = 64);

-- 5 ulasan negatif paling mirip dengan vektor query ($1 dikirim dari Python)
SELECT id, teks, 1 - (embedding <=> $1) AS similarity
FROM ulasan
WHERE label = 'negative' AND n_kata >= 10
ORDER BY embedding <=> $1
LIMIT 5;
```

Operator pgvector: `<->` jarak L2, `<#>` inner product (negatif), `<=>` jarak cosine.

## 5. Mini Aplikasi: Pencarian Ulasan

Gabungkan semuanya menjadi satu fungsi yang siap dipakai aplikasi: query bebas + filter opsional → hasil berurutan. Di modul 09 (RAG), hasil pencarian seperti ini akan diberikan ke LLM sebagai konteks untuk menjawab pertanyaan.

In [ ]:
def cari_ulasan(query, k=5, label=None, min_kata=None):
    hasil = cari_qdrant(query, k=k, label=label, min_kata=min_kata)
    return pd.DataFrame([{"skor": round(p.score, 3), "label": p.payload["label"], "teks": p.payload["text"]} for p in hasil])


print("Ulasan negatif tentang pengiriman:")
print(cari_ulasan("pengiriman barang terlambat", k=5, label="negative").to_string(index=False))

## Ringkasan

- Vector DB = index ANN (umumnya HNSW) + penyimpanan teks & **metadata** + **filter** + CRUD + persistensi.
- **Chroma**: `PersistentClient`, collection dengan `documents`/`metadatas`, filter `where`; mengembalikan **jarak**.
- **Qdrant**: point + payload, filter `must`/`should`; mengembalikan **skor similarity**; mode lokal dan server dengan API yang sama.
- **pgvector**: vektor di PostgreSQL, pencarian lewat SQL (`ORDER BY embedding <=> q`).
- Kunci API database (jika memakai layanan cloud) disimpan di `.env`/Secrets.

## Latihan

1. **Filter vs hasil.** Untuk query `"makanan enak"`, bandingkan hasil tanpa filter, dengan `label="negative"`, dan dengan `min_kata=25`. Jelaskan mengapa filter bisa menghasilkan dokumen dengan similarity rendah, dan bagaimana aplikasi sebaiknya menangani kasus "tidak ada hasil yang cukup relevan" (petunjuk: ambang skor minimum).
2. **Chunking dokumen panjang.** Ambil 3 artikel panjang (misalnya dari Wikipedia bahasa Indonesia), potong menjadi *chunk* ±100 kata dengan overlap 20 kata, simpan ke Chroma dengan metadata `judul` dan `posisi_chunk`, lalu buat fungsi pencarian yang mengembalikan chunk terbaik beserta judulnya. Ini persis langkah *ingest* pada RAG (modul 09).
3. **pgvector sungguhan.** Jalankan PostgreSQL + pgvector (misalnya lewat Docker `pgvector/pgvector` di komputer lokal, atau layanan cloud gratis). Simpan connection string di `.env`, masukkan 1.000 embedding NusaX dengan `psycopg`, lalu jalankan query SQL di bagian 4. Bandingkan hasilnya dengan Qdrant.